# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Dataset Exploration with `mlcroissant`
This notebook provides a demonstration for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure mlcroissant is installed (uncomment below in fresh environments)
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access and display metadata (metadata fields from the Croissant schema)
meta = dataset.metadata
print('Dataset Name: ', meta.name)
print('Identifier:', getattr(meta, 'identifier', None))
print('Version:', getattr(meta, 'version', None))
print('Date Published:', getattr(meta, 'datePublished', None))
print('License:', getattr(meta, 'license', None))
print('Description:', meta.description)
print('Keywords:', getattr(meta, 'keywords', None))

## 2. Data Overview
Review available record sets and their fields. In `mlcroissant`, each record set, field, and column is uniquely identified by an `@id`. We'll enumerate all record sets in the dataset, get their `@id` values, and preview available fields (columns) within one record set.

In [ ]:
# List all record sets and their @ids
record_sets = dataset.record_sets # list of RecordSet objects
if not record_sets:
    print("No record sets detected in the schema (record sets may be discovered via distribution). Fallback to available tables.")
    # mlcroissant datasets can synthesize record sets from files if schema uses files directly.
    # So we derive available record sets from the files/datasets if not in metadata.
    data_tables = list(dataset.available_tables)
    print("Available tables (parsed as record sets):")
    for table in data_tables:
        print(f"  @id: {table}")
    # For demonstration, pick the first table as 'primary' record set
    main_record_set_id = data_tables[0]
else:
    print("Record sets found in schema:")
    for rs in record_sets:
        print(f"  @id: {rs.id}, name: {rs.name if hasattr(rs, 'name') else '-'}")
    # For further steps, use the first record set
    main_record_set_id = record_sets[0].id

print(f"\nPrimary record set to explore: {main_record_set_id}")
# List available columns/fields in the main record set (using @id)
columns = dataset.fields(main_record_set_id)
print(f"Fields (columns) in record set '{main_record_set_id}':")
for field in columns:
    print(f"  @id: {field.id}, name: {field.name if hasattr(field, 'name') else '-'}, type: {getattr(field, 'dataType', '-')}")

## 3. Data Extraction
Load data from the main record set into a Pandas DataFrame for exploration. We use the record set's `@id` and field `@id`s as identifiers.

In [ ]:
# Load records from the main record set using its @id
records = list(dataset.records(record_set=main_record_set_id))
df = pd.DataFrame(records)

print(f"Loaded DataFrame columns (@ids):\n{list(df.columns)}\n")
df.head()

## 4. Exploratory Data Analysis (EDA)
We demonstrate common data processing steps using field `@id`s.

- Filter for records by a numeric column (example: Age > threshold, if present)
- Normalize a numeric column
- Group by a key/categorical column, if present

See the columns printed above for available `@id`s. If "Age" or other suitable numeric fields are available, they will be used.

In [ ]:
# Attempt to identify a numeric field, e.g. one containing "age" or "interval"
import re

# Try simple heuristics to pick columns
numeric_col_candidates = [c for c in df.columns if re.search(r'(age|interval|years|count|score|number|duration)', c, re.IGNORECASE)]
if not numeric_col_candidates:
    numeric_field_id = df.select_dtypes(include=['number']).columns[0] if len(df.select_dtypes(include=['number']).columns) > 0 else df.columns[0]
    print(f"No obvious numeric field detected. Proceeding with: {numeric_field_id}")
else:
    numeric_field_id = numeric_col_candidates[0]
    print(f"Using detected numeric field: {numeric_field_id}")

# Threshold for demonstration (e.g. Age > 50), or use 1 if field is not interpretable
threshold = 50
if numeric_field_id in df.columns:
    # Attempt conversion to numeric if not already
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with '{numeric_field_id}' (as @id) > {threshold}:")
    print(filtered_df.head())
    
    # Normalization
    mean = filtered_df[numeric_field_id].mean()
    std = filtered_df[numeric_field_id].std()
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean) / (std if std != 0 else 1)
    print(f"\nNormalized '{numeric_field_id}' for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by a categorical field, e.g. one including 'sex', 'msi', 'location', etc.
    group_col_candidates = [c for c in df.columns if re.search(r'(sex|msi|location|category|group|site|distribution|status|type)', c, re.IGNORECASE)]
    if group_col_candidates:
        group_field_id = group_col_candidates[0]
        print(f"\nGrouping by '{group_field_id}' (as @id):")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index().rename({numeric_field_id: f'{numeric_field_id}_mean'}, axis=1)
        print(grouped_df.head())
    else:
        print("No suitable categorical field found for grouping in this record set.")
else:
    print("Numeric field not found in DataFrame columns.")

## 5. Visualization
Let's visualize the distribution of the selected numeric field (`@id`: {numeric_field_id}), and, if available, compare it across a categorical group (`@id`, e.g. sample site or status).

_Note: Visualization may require `matplotlib` or `seaborn`. If not installed, uncomment and install them._

In [ ]:
# If matplotlib/seaborn not installed, run:
# !pip install matplotlib seaborn
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=15)
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If grouping field was identified in EDA, do a boxplot across groups
    if 'group_field_id' in locals() and group_field_id in df.columns:
        plt.figure(figsize=(10, 6))
        order = df[group_field_id].dropna().unique()
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df, order=order)
        plt.title(f"'{numeric_field_id}' by '{group_field_id}'")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("Numeric field not available for visualization.")

## 6. Conclusion

In this notebook, we've demonstrated how to:
- Load and inspect a FAIR² dataset using the Croissant specification and `mlcroissant`
- Access all record sets and reference fields/columns by their `@id`
- Load records into Pandas DataFrames for programmatic exploration
- Apply EDA steps (filtering, normalization, grouping) and visualize using common Python libraries

For more in-depth modeling or domain-specific analysis, continue working with these DataFrames and the structured `@id`-indexed schema presented here.

_Tip: Always refer to entities by their `@id` for robust, schema-compliant analysis with Croissant-powered datasets._